# SMTM Multi-Asset All-In-One (SMTM_AIO)

**목표**: KRW 전체 마켓에서 최근 10분간 1분봉 종가가 **지속 상승** 중인 종목만
상승비율 순으로 선별하여 자동 매매하는 멀티에셋 전략의 전체 데이터 흐름을 보여준다.

## 전체 데이터 흐름

```
[Upbit /market/all]
        ↓  전체 KRW 마켓 목록
[가격 필터 MIN_PRICE ~ MAX_PRICE]
        ↓  가격 조건 충족 종목
[10분봉 1분봉 캔들 조회]
        ↓  closing_price 리스트 (10개)
[지속 상승 필터] all(p[i] > p[i-1])
        ↓  상승비율 rise_ratio 계산
[상승비율 내림차순 정렬 → 상위 TOP_N]
        ↓  primary_candle 리스트 (rise_ratio, rank 포함)
[StrategyKRWRising.update_trading_info()]
        ↓  랭킹 갱신, 현재가 캐시
[StrategyKRWRising.get_request()]
        ↓  buy / sell 요청 리스트
[MockMultiTrader.send_request()]
        ↓  즉시 체결 → callback
[StrategyKRWRising.update_result()]
        ↓  잔고 / 보유량 반영
[결과 분석 및 시각화]
```

## 섹션 구성
| 섹션 | 내용 |
|------|------|
| 0 | 환경 설정 (imports, 로깅) |
| 1 | KRW 전체 마켓 조회 및 가격 필터 |
| 2 | 10분간 지속 상승 필터 + 상승비율 랭킹 시각화 |
| 3 | `KRWRisingDataProvider` 구현 및 테스트 |
| 4 | `StrategyKRWRising` 구현 및 초기화 |
| 5 | `MockMultiTrader` 구현 및 테스트 |
| 6 | 트레이딩 루프 — 1턴 단계별 + 다수 턴 시뮬레이션 |
| 7 | 결과 분석 및 시각화 |

> **주의**: `DEMO_MODE = True` 이면 API 호출 없이 합성 데이터로 실행 가능.
> 실시간 데이터를 보려면 `DEMO_MODE = False` 로 변경하세요.


In [ ]:
%matplotlib inline
import sys
import os
import logging
import copy
import math
import time
import random
from datetime import datetime, timedelta

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

# ── 프로젝트 루트를 sys.path 에 추가 ─────────────────────────────────────────
# 노트북이 프로젝트 루트(smtm2/)에 있다고 가정
_ROOT = os.path.abspath(".")
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

# ── smtm 모듈 (analyzer 제외 — matplotlib.use("Agg") 충돌 방지) ─────────────
from smtm.date_converter import DateConverter
from smtm.strategy.strategy import Strategy

# ── 전역 파라미터 ─────────────────────────────────────────────────────────────
DEMO_MODE      = True     # True: 합성 데이터 / False: 실시간 Upbit API
MIN_PRICE      = 100      # 가격 하한 (KRW)
MAX_PRICE      = 100_000  # 가격 상한 (KRW)
CANDLE_COUNT   = 10       # 지속 상승 확인할 1분봉 수
TOP_N          = 15       # 상위 반환 종목 수
INITIAL_BUDGET = 300_000  # 초기 예산 (KRW)
N_TURNS        = 5        # 시뮬레이션 턴 수

print(f"DEMO_MODE      : {DEMO_MODE}")
print(f"가격 필터       : {MIN_PRICE:,} ~ {MAX_PRICE:,} KRW")
print(f"지속 상승 확인   : {CANDLE_COUNT}분봉")
print(f"상위 반환 종목   : {TOP_N}개")
print(f"초기 예산        : {INITIAL_BUDGET:,} KRW")
print(f"시뮬레이션 턴    : {N_TURNS}턴")
print(f"프로젝트 루트    : {_ROOT}")


In [ ]:
class NbLogCapture(logging.Handler):
    """
    셀 실행 중 발생하는 로그를 내부에 누적.
    에러 발생 시 flush_on_error() 를 호출하면 전체 로그를 출력한다.
    """
    def __init__(self):
        super().__init__()
        self.records = []
        self.setFormatter(
            logging.Formatter("[%(levelname)s] %(name)s: %(message)s")
        )

    def emit(self, record):
        self.records.append(self.format(record))

    def flush_on_error(self, prefix="에러 발생 시 수집된 로그"):
        if self.records:
            print("\n" + "=" * 64)
            print(f" {prefix}")
            print("=" * 64)
            for r in self.records:
                print(r)
            print("=" * 64)
        self.records.clear()

    def clear(self):
        self.records.clear()


# 전역 핸들러
nb_log = NbLogCapture()
nb_log.setLevel(logging.DEBUG)

# 루트 로거: WARNING 이상만 콘솔 출력
logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True,
)

print("로깅 설정 완료")
print("  - 루트 로거: WARNING 이상만 출력")
print("  - nb_log 핸들러: DEBUG 이상 누적, 에러 시 flush_on_error() 로 출력")


## 섹션 1 — KRW 전체 마켓 조회 및 가격 필터

**데이터 흐름**

```
Upbit /v1/market/all  →  전체 KRW 마켓 목록
Upbit /v1/ticker      →  현재가
가격 필터 적용         →  price_filtered  {market: price}
```

| API | 설명 |
|-----|------|
| `GET /v1/market/all` | 업비트 전체 마켓 코드 목록 |
| `GET /v1/ticker?markets=...` | 현재가 (최대 100개 배치) |


In [ ]:
# ── 데모용 샘플 마켓 목록 ────────────────────────────────────────────────────
DEMO_MARKETS_BASE = {
    "KRW-XRP":   850.0,   "KRW-DOGE":  215.0,  "KRW-ADA":   680.0,
    "KRW-TRX":   195.0,   "KRW-MATIC": 1100.0, "KRW-DOT":   8500.0,
    "KRW-LINK":  16000.0, "KRW-ATOM":  9200.0, "KRW-SAND":   530.0,
    "KRW-MANA":  490.0,   "KRW-THETA": 2300.0, "KRW-GRT":    310.0,
    "KRW-BTT":   0.18,    "KRW-SOL":   180000.0,               # 가격 필터 밖
    "KRW-STX":   1850.0,  "KRW-FLOW":  950.0,  "KRW-IMX":   2100.0,
    "KRW-NEAR":  4200.0,  "KRW-ALGO":  310.0,  "KRW-VET":    55.0,
}

UPBIT_MARKET_ALL_URL = "https://api.upbit.com/v1/market/all"
UPBIT_TICKER_URL     = "https://api.upbit.com/v1/ticker"
UPBIT_CANDLE_URL     = "https://api.upbit.com/v1/candles/minutes/1"

nb_log.clear()
try:
    if DEMO_MODE:
        all_krw_markets = list(DEMO_MARKETS_BASE.keys())
        print(f"[DEMO] 샘플 마켓 {len(all_krw_markets)}개 사용")
    else:
        resp = requests.get(
            UPBIT_MARKET_ALL_URL,
            params={"isDetails": False},
            timeout=10,
        )
        resp.raise_for_status()
        all_krw_markets = [
            m["market"] for m in resp.json()
            if m["market"].startswith("KRW-")
        ]
        print(f"Upbit KRW 전체 마켓: {len(all_krw_markets)}개")

    print(f"첫 10개 샘플: {all_krw_markets[:10]}")

except requests.exceptions.RequestException as e:
    nb_log.flush_on_error("마켓 목록 API 조회 실패")
    raise
except Exception as e:
    nb_log.flush_on_error("마켓 목록 처리 오류")
    raise


In [ ]:
nb_log.clear()
try:
    if DEMO_MODE:
        current_prices = dict(DEMO_MARKETS_BASE)
        print(f"[DEMO] 샘플 현재가 {len(current_prices)}개 사용")
    else:
        current_prices = {}
        batch_size = 100
        for i in range(0, len(all_krw_markets), batch_size):
            batch = all_krw_markets[i : i + batch_size]
            resp = requests.get(
                UPBIT_TICKER_URL,
                params={"markets": ",".join(batch)},
                timeout=10,
            )
            resp.raise_for_status()
            for item in resp.json():
                current_prices[item["market"]] = float(item["trade_price"])
            time.sleep(0.1)  # 레이트 리밋 방지
        print(f"현재가 조회 완료: {len(current_prices)}개")

    # ── 가격 필터 적용 ──────────────────────────────────────────────────────
    price_filtered = {
        m: p
        for m, p in current_prices.items()
        if MIN_PRICE <= p <= MAX_PRICE
    }

    print(f"\n전체 마켓       : {len(current_prices)}개")
    print(f"가격 필터 조건   : {MIN_PRICE:,} ~ {MAX_PRICE:,} KRW")
    print(f"가격 필터 후     : {len(price_filtered)}개")
    print(f"제외된 마켓      : {len(current_prices) - len(price_filtered)}개")

    # 가격 분포 요약
    df_prices = pd.DataFrame(
        [(m.replace("KRW-", ""), p) for m, p in
         sorted(price_filtered.items(), key=lambda x: -x[1])],
        columns=["심볼", "현재가(KRW)"],
    )
    print("\n상위 10개 (현재가 내림차순):")
    display(df_prices.head(10).to_string(index=False))

except requests.exceptions.RequestException as e:
    nb_log.flush_on_error("현재가 API 조회 실패")
    raise
except Exception as e:
    nb_log.flush_on_error("가격 필터 처리 오류")
    raise


## 섹션 2 — 10분간 지속 상승 필터 + 상승비율 랭킹

**판별 기준**

```
candles = [c0, c1, c2, ..., c9]   # 1분봉 종가, 시간순
지속 상승 ↔ all(c[i] > c[i-1] for i in 1..9)
상승비율   = (c9 - c0) / c0
```

**데이터 흐름**

```
price_filtered  →  Upbit /v1/candles/minutes/1?count=10 (종목별)
                →  is_continuously_rising() 판별
                →  calc_rise_ratio() 계산
                →  내림차순 정렬  →  rising_assets 리스트
```


In [ ]:
def _fetch_real_candles(market, count=10):
    """Upbit 1분봉 캔들 종가 리스트 반환 (시간순, 오래된 것부터)."""
    try:
        resp = requests.get(
            UPBIT_CANDLE_URL,
            params={"market": market, "count": count},
            timeout=5,
        )
        resp.raise_for_status()
        data = resp.json()
        data.reverse()          # 업비트: 최신순 → 시간순
        return [float(c["trade_price"]) for c in data]
    except Exception as e:
        logging.getLogger("fetch_candles").warning(f"{market}: {e}")
        return []


def _make_demo_candles(base_price, count=10, seed=None):
    """합성 1분봉 종가 리스트 생성. seed 가 None 이면 매번 달라짐."""
    rng = random.Random(seed)
    prices = [base_price]
    for _ in range(count - 1):
        delta = rng.uniform(-0.004, 0.009) * prices[-1]
        prices.append(round(prices[-1] + delta, 4))
    return prices


def is_continuously_rising(prices):
    """종가 리스트가 단조 증가인지 확인 (strict)."""
    if len(prices) < 2:
        return False
    return all(prices[i] > prices[i - 1] for i in range(1, len(prices)))


def calc_rise_ratio(prices):
    """상승비율 = (마지막 종가 - 첫 종가) / 첫 종가."""
    if not prices or prices[0] == 0:
        return 0.0
    return (prices[-1] - prices[0]) / prices[0]


# ── 기능 검증 ─────────────────────────────────────────────────────────────
sample_rising    = [100, 101, 102, 103, 104, 105, 106, 107, 108, 109]
sample_non_rising = [100, 101, 100, 103, 104, 105, 106, 107, 108, 109]

print("is_continuously_rising 검증:")
print(f"  단조증가 리스트: {is_continuously_rising(sample_rising)}")   # True
print(f"  중간 하락 리스트: {is_continuously_rising(sample_non_rising)}")  # False
print(f"  calc_rise_ratio: {calc_rise_ratio(sample_rising)*100:.4f}%")


In [ ]:
print(f"총 {len(price_filtered)}개 종목 대상으로 10분봉 지속 상승 탐색 중...")
if not DEMO_MODE:
    print("(Upbit API 호출 진행 중... 약간의 시간이 소요됩니다)")

nb_log.clear()
rising_assets = []

try:
    for idx, (market, base_price) in enumerate(price_filtered.items()):
        if DEMO_MODE:
            candles = _make_demo_candles(base_price)
        else:
            candles = _fetch_real_candles(market, CANDLE_COUNT)
            time.sleep(0.07)       # Upbit: ~10 req/sec

        if not candles:
            continue

        if is_continuously_rising(candles):
            ratio = calc_rise_ratio(candles)
            rising_assets.append({
                "market":       market,
                "symbol":       market.replace("KRW-", ""),
                "first_price":  candles[0],
                "current_price": candles[-1],
                "rise_ratio":   ratio,
                "candles":      candles,
            })

        if (idx + 1) % 10 == 0:
            print(f"  {idx + 1}/{len(price_filtered)} 처리 완료...")

    # 상승비율 내림차순 정렬
    rising_assets.sort(key=lambda x: x["rise_ratio"], reverse=True)
    for rank, a in enumerate(rising_assets, start=1):
        a["rank"] = rank

    print(f"\n탐색 완료")
    print(f"  지속 상승 종목 : {len(rising_assets)}개 / {len(price_filtered)}개")

    if rising_assets:
        df_rising = pd.DataFrame([{
            "순위":      a["rank"],
            "종목":      a["symbol"],
            "첫 종가":   f"{a['first_price']:,.2f}",
            "현재 종가": f"{a['current_price']:,.2f}",
            "상승비율":  f"{a['rise_ratio']*100:.4f}%",
        } for a in rising_assets[:20]])
        print("\n상위 20개:")
        display(df_rising.to_string(index=False))
    else:
        print("  현재 조건을 만족하는 종목이 없습니다.")
        print("  DEMO_MODE=True 이면 합성 데이터의 랜덤성으로 인해 없을 수 있습니다.")
        print("  다시 실행하거나 MIN_PRICE/MAX_PRICE 조건을 완화하세요.")

except Exception as e:
    nb_log.flush_on_error("지속 상승 탐색 중 오류")
    raise


In [ ]:
if rising_assets:
    top_n_viz = min(TOP_N, len(rising_assets))
    top = rising_assets[:top_n_viz]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("10분간 지속 상승 종목 — 필터링 결과", fontsize=13, fontweight="bold")

    # 왼쪽: 상승비율 수평 막대차트
    ax1 = axes[0]
    symbols = [a["symbol"] for a in reversed(top)]
    ratios  = [a["rise_ratio"] * 100 for a in reversed(top)]
    max_r   = max(ratios) if ratios else 1
    colors  = plt.cm.RdYlGn([r / max_r for r in ratios])
    bars = ax1.barh(symbols, ratios, color=colors, edgecolor="white", linewidth=0.5)
    ax1.set_xlabel("상승비율 (%)")
    ax1.set_title(f"상위 {top_n_viz}종목 상승비율")
    ax1.grid(axis="x", alpha=0.3)
    for bar, ratio in zip(bars, ratios):
        ax1.text(
            bar.get_width() + max_r * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{ratio:.4f}%",
            va="center",
            fontsize=8,
        )

    # 오른쪽: 상위 5종목 10분간 가격 변화율 추이
    ax2 = axes[1]
    top5 = rising_assets[:5]
    for asset in top5:
        base = asset["candles"][0]
        norm = [(p - base) / base * 100 for p in asset["candles"]]
        ax2.plot(range(len(norm)), norm, marker="o", markersize=3,
                 label=asset["symbol"])
    ax2.set_xlabel("분봉 인덱스 (0=10분 전, 9=현재)")
    ax2.set_ylabel("변화율 (%)")
    ax2.set_title("상위 5종목 10분간 종가 변화율")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)
    ax2.axhline(0, color="black", linestyle="--", linewidth=0.5)

    plt.tight_layout()
    os.makedirs("output", exist_ok=True)
    plt.savefig("output/s2_rising_rank.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("차트 저장: output/s2_rising_rank.png")
else:
    print("시각화할 데이터 없음 — 상승 종목이 탐지되지 않았습니다.")


## 섹션 3 — `KRWRisingDataProvider` 구현 및 테스트

위 섹션 1~2의 로직을 `get_info()` 한 번의 호출로 캡슐화한 데이터 프로바이더.

**반환 스키마** (`primary_candle` + 추가 필드)

```python
{
    "type":          "primary_candle",
    "market":        "XRP",          # currency 코드 (KRW- 제거)
    "market_full":   "KRW-XRP",
    "date_time":     "2025-03-15T12:00:00",
    "opening_price": 840.0,          # 10분 전 첫 종가
    "high_price":    860.0,          # 10분간 최고가
    "low_price":     835.0,          # 10분간 최저가
    "closing_price": 858.0,          # 현재 종가
    "acc_price":     0.0,            # 미사용 (인터페이스 충족용)
    "acc_volume":    0.0,            # 미사용
    "rise_ratio":    0.0214,         # 상승비율 (소수)
    "rank":          1,              # 상승비율 순위
}
```


In [ ]:
class KRWRisingDataProvider:
    """
    Upbit KRW 전체 마켓에서 최근 CANDLE_COUNT 분봉 동안 종가가
    지속 상승한 종목을 탐색하여 상승비율 순으로 반환하는 데이터 프로바이더.

    데이터 흐름:
        [Upbit /market/all]
            → [가격 필터 MIN_PRICE ~ MAX_PRICE]
            → [1분봉 캔들 × CANDLE_COUNT 조회 (종목별)]
            → [is_continuously_rising() 판별]
            → [calc_rise_ratio() 계산 → 내림차순 정렬]
            → [상위 TOP_N → primary_candle 리스트 반환]
    """

    CODE = "KRWRISE"
    NAME = "KRW Rising DataProvider"

    MARKET_ALL_URL = "https://api.upbit.com/v1/market/all"
    TICKER_URL     = "https://api.upbit.com/v1/ticker"
    CANDLE_1M_URL  = "https://api.upbit.com/v1/candles/minutes/1"

    def __init__(
        self,
        min_price: float   = 100,
        max_price: float   = 100_000,
        candle_count: int  = 10,
        top_n: int         = 20,
        demo_mode: bool    = False,
        demo_base_prices: dict = None,   # {market: base_price}
    ):
        self.min_price    = min_price
        self.max_price    = max_price
        self.candle_count = candle_count
        self.top_n        = top_n
        self.demo_mode    = demo_mode
        self.demo_base    = demo_base_prices or {}

        self.logger = logging.getLogger(self.__class__.__name__)
        self.logger.addHandler(nb_log)
        self.logger.setLevel(logging.DEBUG)

    # ── 내부 API 헬퍼 ─────────────────────────────────────────────────────

    def _all_krw_markets(self):
        if self.demo_mode:
            return list(self.demo_base.keys())
        try:
            r = requests.get(self.MARKET_ALL_URL,
                             params={"isDetails": False}, timeout=10)
            r.raise_for_status()
            return [m["market"] for m in r.json()
                    if m["market"].startswith("KRW-")]
        except Exception as e:
            self.logger.error(f"마켓 목록 조회 실패: {e}")
            return []

    def _get_prices(self, markets):
        if self.demo_mode:
            return {m: p for m, p in self.demo_base.items() if m in markets}
        prices = {}
        for i in range(0, len(markets), 100):
            batch = markets[i : i + 100]
            try:
                r = requests.get(self.TICKER_URL,
                                 params={"markets": ",".join(batch)},
                                 timeout=10)
                r.raise_for_status()
                for item in r.json():
                    prices[item["market"]] = float(item["trade_price"])
                time.sleep(0.1)
            except Exception as e:
                self.logger.error(f"현재가 배치 조회 실패 (idx={i}): {e}")
        return prices

    def _get_candles(self, market, base_price=None):
        if self.demo_mode and base_price is not None:
            prices = [base_price]
            rng = random.Random()
            for _ in range(self.candle_count - 1):
                delta = rng.uniform(-0.004, 0.009) * prices[-1]
                prices.append(round(prices[-1] + delta, 4))
            return prices
        try:
            r = requests.get(self.CANDLE_1M_URL,
                             params={"market": market,
                                     "count": self.candle_count},
                             timeout=5)
            r.raise_for_status()
            data = r.json()
            data.reverse()          # 업비트: 최신순 → 시간순
            return [float(c["trade_price"]) for c in data]
        except Exception as e:
            self.logger.error(f"캔들 조회 실패 ({market}): {e}")
            return []

    @staticmethod
    def _is_rising(prices):
        if len(prices) < 2:
            return False
        return all(prices[i] > prices[i - 1] for i in range(1, len(prices)))

    @staticmethod
    def _rise_ratio(prices):
        if not prices or prices[0] == 0:
            return 0.0
        return (prices[-1] - prices[0]) / prices[0]

    # ── 공개 인터페이스 ────────────────────────────────────────────────────

    def get_info(self):
        """
        KRW 마켓 전체를 탐색하여 조건을 만족하는 종목의
        primary_candle 리스트를 상승비율 순으로 반환한다.

        반환 필드 (primary_candle + 확장):
            type, market, market_full, date_time,
            opening_price, high_price, low_price, closing_price,
            acc_price(0), acc_volume(0),
            rise_ratio, rank
        """
        self.logger.info("[get_info] 시작")

        # Step 1: 전체 KRW 마켓 목록
        all_markets = self._all_krw_markets()
        self.logger.info(f"  전체 마켓: {len(all_markets)}개")

        # Step 2: 현재가 조회 + 가격 필터
        prices = self._get_prices(all_markets)
        filtered = [
            m for m in all_markets
            if self.min_price <= prices.get(m, 0) <= self.max_price
        ]
        self.logger.info(
            f"  가격 필터 ({self.min_price:,.0f}~{self.max_price:,.0f}): "
            f"{len(filtered)}개"
        )

        # Step 3: 10분봉 지속 상승 확인 + 상승비율
        rising = []
        for market in filtered:
            base = prices.get(market)
            candles = self._get_candles(market, base)
            if not candles:
                continue
            if self._is_rising(candles):
                rising.append({
                    "market":  market,
                    "candles": candles,
                    "ratio":   self._rise_ratio(candles),
                })
            if not self.demo_mode:
                time.sleep(0.07)

        # Step 4: 정렬 + 상위 TOP_N
        rising.sort(key=lambda x: x["ratio"], reverse=True)
        top = rising[: self.top_n]
        self.logger.info(
            f"  지속 상승 탐지: {len(rising)}개 → 상위 {len(top)}개 반환"
        )

        # Step 5: primary_candle 형식 변환
        now = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
        result = []
        for rank, item in enumerate(top, start=1):
            c = item["candles"]
            result.append({
                "type":          "primary_candle",
                "market":        item["market"].replace("KRW-", ""),
                "market_full":   item["market"],
                "date_time":     now,
                "opening_price": c[0],
                "high_price":    max(c),
                "low_price":     min(c),
                "closing_price": c[-1],
                "acc_price":     0.0,
                "acc_volume":    0.0,
                "rise_ratio":    item["ratio"],
                "rank":          rank,
            })

        return result


In [ ]:
nb_log.clear()
try:
    dp = KRWRisingDataProvider(
        min_price    = MIN_PRICE,
        max_price    = MAX_PRICE,
        candle_count = CANDLE_COUNT,
        top_n        = TOP_N,
        demo_mode    = DEMO_MODE,
        demo_base_prices = price_filtered if DEMO_MODE else None,
    )

    print("KRWRisingDataProvider.get_info() 호출...")
    t0 = time.time()
    candle_info = dp.get_info()
    elapsed = time.time() - t0

    print(f"\n반환 캔들 수: {len(candle_info)}개  (소요: {elapsed:.2f}초)")

    if candle_info:
        print("\n첫 번째 캔들 딕셔너리 (스키마 확인):")
        for k, v in candle_info[0].items():
            print(f"  {k:<20s}: {v}")

        print("\n상위 10종목 요약:")
        for item in candle_info[:10]:
            print(
                f"  #{item['rank']:2d}  {item['market']:<6s}  "
                f"종가: {item['closing_price']:>10,.2f} KRW  "
                f"상승비율: {item['rise_ratio']*100:.4f}%"
            )
    else:
        print("반환된 종목 없음 — DEMO_MODE 에서 합성 데이터가 우연히 단조 증가를")
        print("만족하지 않는 경우입니다. 셀을 다시 실행하세요.")

except Exception as e:
    nb_log.flush_on_error("데이터 프로바이더 테스트 실패")
    raise


## 섹션 4 — `StrategyKRWRising` 구현

`KRWRisingDataProvider.get_info()` 가 반환한 **rise_ratio + rank** 정보를
활용하는 멀티에셋 전략.

**매수 조건**
- 상위 `MAX_BUY_COUNT` 종목에 포함
- 보유 중이지 않고 매수 대기 주문도 없음
- 잔고 ≥ `min_price`

**매도 조건**
1. **추세 소멸**: 다음 턴에 지속 상승 목록에서 이탈
2. **손절**: 매수 평균가 대비 `STOP_LOSS_RATIO`% 이상 하락

**요청 딕셔너리 구조**

```python
{
    "id":        "1741234567.890123",   # DateConverter.timestamp_id()
    "type":      "buy" | "sell",
    "market":    "XRP",                 # currency 코드 (UpbitMultiTrader 라우팅용)
    "price":     858.0,
    "amount":    58.3205,
    "date_time": "2025-03-15T12:01:00",
}
```


In [ ]:
class StrategyKRWRising(Strategy):
    """
    KRW 전체 마켓 중 10분간 지속 상승 종목에 투자하는 멀티에셋 전략.

    update_trading_info() 에서 KRWRisingDataProvider 의 rise_ratio / rank 를
    활용하여 rankings 를 갱신하고, get_request() 에서 매수/매도 요청을 생성한다.
    """

    NAME = "KRW Rising Multi"
    CODE = "KRM"

    MAX_BUY_COUNT    = 3          # 동시 보유 최대 종목 수
    MAX_BUY_AMOUNT   = 50_000    # 종목당 최대 매수 금액 (KRW)
    STOP_LOSS_RATIO  = 0.03      # 손절 기준 하락률 (3%)
    COMMISSION_RATIO = 0.0005    # 거래 수수료율

    def __init__(self):
        self.is_initialized  = False
        self.budget          = 0
        self.balance         = 0
        self.min_price       = 0

        # {currency: {"amount": float, "avg_price": float, "current_price": float}}
        self.holdings        = {}

        # [(currency, rise_ratio), ...]  현재 턴 랭킹 (rise_ratio 내림차순)
        self.rankings        = []

        # {currency: float}  최신 종가 캐시 (get_request 에서 사용)
        self._last_prices    = {}

        # {req_id: {"market": currency, "type": str}}  대기 중인 주문
        self.waiting_requests = {}

        self.result          = []
        self.logger = logging.getLogger(self.__class__.__name__)
        self.logger.addHandler(nb_log)
        self.logger.setLevel(logging.DEBUG)

    # ── Strategy 추상 메서드 구현 ──────────────────────────────────────────

    def initialize(self, budget, min_price=1000,
                   add_spot_callback=None,
                   add_line_callback=None,
                   alert_callback=None):
        """예산 및 최소 거래 금액 설정. 한 번만 유효."""
        if self.is_initialized:
            return
        self.budget      = budget
        self.balance     = budget
        self.min_price   = min_price
        self.is_initialized = True
        self.logger.info(
            f"[{self.NAME}] 초기화: budget={budget:,}, min_price={min_price}, "
            f"max_buy_count={self.MAX_BUY_COUNT}, "
            f"stop_loss={self.STOP_LOSS_RATIO*100:.1f}%"
        )

    def update_trading_info(self, info):
        """
        KRWRisingDataProvider.get_info() 반환값으로 랭킹과 현재가를 갱신한다.

        info: primary_candle 리스트 (rise_ratio, rank 필드 포함)
        """
        if not self.is_initialized:
            return

        new_rankings = []
        for item in info:
            if not isinstance(item, dict):
                continue
            if item.get("type") != "primary_candle":
                continue

            currency = item["market"]
            price    = float(item.get("closing_price", 0))
            ratio    = float(item.get("rise_ratio", 0))

            # 최신 종가 캐시 갱신
            self._last_prices[currency] = price

            # 보유 종목이면 현재가도 갱신 (손절 판단용)
            if currency in self.holdings:
                self.holdings[currency]["current_price"] = price

            new_rankings.append((currency, ratio))

        # rise_ratio 내림차순 정렬
        new_rankings.sort(key=lambda x: x[1], reverse=True)
        self.rankings = new_rankings

        self.logger.info(
            "[update_trading_info] 랭킹 갱신: "
            + str([(c, f"{r*100:.4f}%") for c, r in self.rankings[:5]])
        )

    def get_request(self):
        """
        1. 손절 / 추세 소멸 종목 → sell 요청
        2. 상위 종목 중 미보유 종목 → buy 요청

        Returns: 요청 딕셔너리 리스트 or None
        """
        if not self.is_initialized or not self.rankings:
            return None

        requests_list = []
        ranked_set    = {c for c, _ in self.rankings}
        pending_sell  = {
            v["market"] for v in self.waiting_requests.values()
            if v["type"] == "sell"
        }
        pending_buy = {
            v["market"] for v in self.waiting_requests.values()
            if v["type"] == "buy"
        }

        # ── 1. 매도 체크 ──────────────────────────────────────────────────
        for currency in list(self.holdings.keys()):
            if currency in pending_sell:
                continue

            holding       = self.holdings[currency]
            current_price = holding.get("current_price", holding["avg_price"])
            stop_price    = holding["avg_price"] * (1 - self.STOP_LOSS_RATIO)

            # 가격 손절
            if current_price < stop_price:
                req = self._create_sell_req(currency, current_price)
                if req:
                    requests_list.append(req)
                    self.logger.info(
                        f"[SELL/stop_loss] {currency} @ {current_price:,.2f} "
                        f"(avg={holding['avg_price']:,.2f}, "
                        f"threshold={stop_price:,.2f})"
                    )
                continue

            # 추세 소멸 (지속 상승 목록 이탈)
            if currency not in ranked_set:
                req = self._create_sell_req(currency, current_price)
                if req:
                    requests_list.append(req)
                    self.logger.info(
                        f"[SELL/trend_over] {currency} @ {current_price:,.2f}"
                    )

        # ── 2. 매수 체크 ──────────────────────────────────────────────────
        active_count = (
            len([c for c in self.holdings if c not in pending_sell])
            + len(pending_buy)
        )

        for currency, ratio in self.rankings:
            if active_count >= self.MAX_BUY_COUNT:
                break
            if currency in self.holdings or currency in pending_buy:
                continue

            current_price = self._last_prices.get(currency, 0)
            if current_price <= 0:
                continue

            buy_budget = min(self.MAX_BUY_AMOUNT, self.balance)
            if buy_budget < self.min_price:
                self.logger.warning(
                    f"잔고 부족으로 매수 중단 (balance={self.balance:,})"
                )
                break

            req = self._create_buy_req(currency, current_price, buy_budget)
            if req:
                requests_list.append(req)
                active_count += 1
                rank_pos = [c for c, _ in self.rankings].index(currency) + 1
                self.logger.info(
                    f"[BUY rank={rank_pos}, rise={ratio*100:.4f}%] "
                    f"{currency} @ {current_price:,.2f}"
                )

        return requests_list if requests_list else None

    def update_result(self, result):
        """체결 결과를 반영하여 잔고 / 보유량을 갱신한다."""
        if not isinstance(result, dict):
            return
        if result.get("state") != "done":
            return

        req        = result.get("request", {})
        req_id     = req.get("id")
        if req_id in self.waiting_requests:
            del self.waiting_requests[req_id]

        currency   = req.get("market")
        trade_type = result.get("type")
        price      = float(result.get("price", 0))
        amount     = float(result.get("amount", 0))
        msg        = result.get("msg", "")

        if not currency or price <= 0 or amount <= 0 or msg != "success":
            return

        total = price * amount
        fee   = total * self.COMMISSION_RATIO

        if trade_type == "buy":
            self.balance -= round(total + fee)
            if currency not in self.holdings:
                self.holdings[currency] = {
                    "amount": 0.0, "avg_price": 0.0, "current_price": price
                }
            old        = self.holdings[currency]
            old_val    = old["avg_price"] * old["amount"]
            new_amount = old["amount"] + amount
            self.holdings[currency] = {
                "amount":        round(new_amount, 6),
                "avg_price":     round((old_val + total) / new_amount, 6),
                "current_price": price,
            }
            self.logger.info(
                f"[BUY done] {currency} {amount:.6f} @ {price:,.2f}  "
                f"balance={self.balance:,}"
            )

        elif trade_type == "sell":
            self.balance += round(total - fee)
            if currency in self.holdings:
                new_amount = round(self.holdings[currency]["amount"] - amount, 6)
                if new_amount <= 0:
                    del self.holdings[currency]
                else:
                    self.holdings[currency]["amount"] = new_amount
            self.logger.info(
                f"[SELL done] {currency} {amount:.6f} @ {price:,.2f}  "
                f"balance={self.balance:,}"
            )

        self.result.append(copy.deepcopy(result))

    # ── Private 헬퍼 ──────────────────────────────────────────────────────

    def _create_buy_req(self, currency, price, budget):
        net    = budget * (1 - self.COMMISSION_RATIO)
        amount = math.floor(net / price * 10000) / 10000
        if amount <= 0:
            return None
        req_id = DateConverter.timestamp_id()
        self.waiting_requests[req_id] = {"market": currency, "type": "buy"}
        return {
            "id":        req_id,
            "type":      "buy",
            "market":    currency,
            "price":     price,
            "amount":    amount,
            "date_time": datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
        }

    def _create_sell_req(self, currency, price):
        holding = self.holdings.get(currency)
        if not holding:
            return None
        amount = math.floor(holding["amount"] * 10000) / 10000
        if amount <= 0:
            return None
        req_id = DateConverter.timestamp_id()
        self.waiting_requests[req_id] = {"market": currency, "type": "sell"}
        return {
            "id":        req_id,
            "type":      "sell",
            "market":    currency,
            "price":     price,
            "amount":    amount,
            "date_time": datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
        }


In [ ]:
nb_log.clear()
try:
    strategy = StrategyKRWRising()
    strategy.initialize(INITIAL_BUDGET, min_price=1000)

    print("StrategyKRWRising 초기화 완료")
    print(f"  이름         : {strategy.NAME}  (CODE={strategy.CODE})")
    print(f"  초기 예산    : {strategy.budget:,} KRW")
    print(f"  최대 보유    : {strategy.MAX_BUY_COUNT}종목")
    print(f"  종목당 매수  : 최대 {strategy.MAX_BUY_AMOUNT:,} KRW")
    print(f"  손절 기준    : {strategy.STOP_LOSS_RATIO*100:.1f}%")
    print(f"  수수료율     : {strategy.COMMISSION_RATIO*100:.2f}%")
    print(f"  초기화 상태  : {strategy.is_initialized}")

except Exception as e:
    nb_log.flush_on_error("전략 초기화 실패")
    raise


## 섹션 5 — `MockMultiTrader` 구현

노트북 전용 모의 트레이더. **실제 API 호출 없이** 지정 가격으로 즉시 체결 처리.

> 실거래에서는 `UpbitMultiTrader` 를 사용하며,
> `request["market"]` 필드로 종목별 sub-trader 에 라우팅한다.

**체결 흐름**

```
send_request([req1, req2, ...], callback)
    ↓  각 요청에 대해
    _fill_order(req)  →  잔고/보유량 갱신
    callback(result)  →  strategy.update_result() 에 전달
```

**result 딕셔너리 구조**

```python
{
    "request":   <원본 요청>,
    "type":      "buy" | "sell" | "cancel",
    "market":    "XRP",
    "price":     858.0,
    "amount":    58.3205,
    "msg":       "success" | "insufficient_balance" | ...,
    "state":     "done",
    "date_time": "2025-03-15T12:01:00",
}
```


In [ ]:
class MockMultiTrader:
    """
    멀티에셋 모의 트레이더 (노트북 전용).

    실제 API 호출 없이 지정 가격으로 즉시 체결 처리한다.
    잔고와 보유량은 내부 상태로 관리하며,
    실거래 시에는 UpbitMultiTrader 로 교체한다.
    """

    NAME             = "MockMultiTrader"
    COMMISSION_RATIO = 0.0005

    def __init__(self, initial_budget: float):
        self.balance   = initial_budget
        self.assets    = {}    # {currency: amount}
        self.trade_log = []    # 체결 내역 [{type, market, price, amount, ...}]
        self.logger    = logging.getLogger(self.__class__.__name__)
        self.logger.addHandler(nb_log)
        self.logger.setLevel(logging.DEBUG)

    def send_request(self, request_list, callback):
        """요청 리스트를 순차적으로 체결 처리하고 callback 을 호출한다."""
        for req in (request_list or []):
            result = self._fill_order(req)
            callback(result)

    def _fill_order(self, req):
        """단일 주문을 즉시 체결 처리하고 결과 딕셔너리를 반환한다."""
        currency   = req.get("market", "UNKNOWN")
        trade_type = req.get("type", "")
        price      = float(req.get("price", 0))
        amount     = float(req.get("amount", 0))
        now        = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")

        base_result = {
            "request":   req,
            "type":      trade_type,
            "price":     price,
            "amount":    amount,
            "state":     "done",
            "date_time": now,
            "market":    currency,
        }

        if trade_type == "cancel":
            return {**base_result, "msg": "success"}

        if price <= 0 or amount <= 0:
            return {**base_result, "msg": "invalid_order"}

        fee   = price * amount * self.COMMISSION_RATIO
        total = price * amount

        if trade_type == "buy":
            cost = total + fee
            if cost > self.balance:
                self.logger.warning(
                    f"[BUY FAIL] {currency} 잔고 부족 "
                    f"(필요={cost:,.0f} / 보유={self.balance:,.0f})"
                )
                return {**base_result, "msg": "insufficient_balance"}
            self.balance -= round(cost)
            self.assets[currency] = round(
                self.assets.get(currency, 0.0) + amount, 6
            )
            msg = "success"

        elif trade_type == "sell":
            have = self.assets.get(currency, 0.0)
            if have < amount:
                self.logger.warning(
                    f"[SELL FAIL] {currency} 보유량 부족 "
                    f"(요청={amount:.6f} / 보유={have:.6f})"
                )
                return {**base_result, "msg": "insufficient_asset"}
            self.balance += round(total - fee)
            self.assets[currency] = round(have - amount, 6)
            msg = "success"

        else:
            return {**base_result, "msg": "unknown_type"}

        self.trade_log.append({
            "type":   trade_type,
            "market": currency,
            "price":  price,
            "amount": amount,
            "total":  round(total),
            "fee":    round(fee),
            "time":   now,
        })
        self.logger.info(
            f"[{trade_type.upper()}] {currency} {amount:.6f} @ {price:,.2f} "
            f"fee={fee:,.0f}  balance={self.balance:,}"
        )
        return {**base_result, "msg": msg}

    def cancel_request(self, request_id):
        pass

    def cancel_all_requests(self):
        pass

    def get_account_info(self):
        return {
            "balance":   self.balance,
            "asset":     {k: v for k, v in self.assets.items() if v > 0},
            "date_time": datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
        }


In [ ]:
nb_log.clear()
try:
    test_trader = MockMultiTrader(initial_budget=300_000)

    # ── 매수 테스트 ────────────────────────────────────────────────────────
    buy_req = {
        "id":        DateConverter.timestamp_id(),
        "type":      "buy",
        "market":    "XRP",
        "price":     850.0,
        "amount":    10.0,
        "date_time": datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
    }
    results = []
    test_trader.send_request([buy_req], lambda r: results.append(r))
    r = results[0]

    print("매수 요청 결과:")
    for k in ("type", "market", "price", "amount", "msg", "state"):
        print(f"  {k:<10}: {r[k]}")
    print(f"  잔고     : {test_trader.balance:,} KRW")
    print(f"  보유     : {test_trader.assets}")

    # ── 매도 테스트 ────────────────────────────────────────────────────────
    sell_req = {
        "id":        DateConverter.timestamp_id(),
        "type":      "sell",
        "market":    "XRP",
        "price":     860.0,
        "amount":    10.0,
        "date_time": datetime.now().strftime("%Y-%m-%dT%H:%M:%S"),
    }
    results2 = []
    test_trader.send_request([sell_req], lambda r: results2.append(r))
    r2 = results2[0]

    print("\n매도 요청 결과:")
    for k in ("type", "market", "price", "amount", "msg", "state"):
        print(f"  {k:<10}: {r2[k]}")
    print(f"  잔고     : {test_trader.balance:,} KRW")
    pnl = round(test_trader.balance - 300_000)
    print(f"  손익     : {pnl:+,} KRW")

except Exception as e:
    nb_log.flush_on_error("트레이더 테스트 실패")
    raise


## 섹션 6 — 트레이딩 루프 (동기 실행)

> **스레딩 없음**: `Operator` / `SimulationOperator` 는 내부적으로 Worker 스레드를
> 사용하지만, 노트북에서는 **`_execute_trading_sync()` 를 직접 호출**하여
> 동기적으로 실행한다. 이렇게 해야 셀 실행 결과를 즉시 확인할 수 있다.

### 핵심 트레이딩 루프 5단계

```
Step 1  DataProvider.get_info()              → trading_info (primary_candle 리스트)
Step 2  Strategy.update_trading_info(info)   → 랭킹 갱신, 현재가 캐시
Step 3  Strategy.get_request()               → requests (buy/sell 리스트)
Step 4  Trader.send_request(requests, cb)    → 체결 + callback 호출
Step 5  (callback 내) Strategy.update_result → 잔고/보유량 반영
```


In [ ]:
def _execute_trading_sync(dp, strategy, trader):
    """
    트레이딩 루프 1턴을 동기적으로 실행한다.
    스레딩 없이 5단계를 순서대로 호출한다.

    Returns: (trading_info, requests, results)
    """
    # Step 1
    trading_info = dp.get_info()

    # Step 2
    strategy.update_trading_info(trading_info)

    # Step 3
    requests = strategy.get_request()

    # Step 4 + Step 5 (callback 내)
    results = []
    if requests:
        def _cb(result):
            strategy.update_result(result)   # Step 5
            results.append(result)
        trader.send_request(requests, _cb)

    return trading_info, requests, results


# ── 1턴 단계별 실행 ──────────────────────────────────────────────────────────
print("=" * 64)
print(" 트레이딩 루프 1턴 — 단계별 실행")
print("=" * 64)

nb_log.clear()
try:
    # 새 인스턴스 생성 (상태 초기화)
    strategy1 = StrategyKRWRising()
    strategy1.initialize(INITIAL_BUDGET, min_price=1000)
    trader1   = MockMultiTrader(initial_budget=INITIAL_BUDGET)
    dp1       = KRWRisingDataProvider(
        min_price=MIN_PRICE, max_price=MAX_PRICE,
        candle_count=CANDLE_COUNT, top_n=TOP_N,
        demo_mode=DEMO_MODE,
        demo_base_prices=price_filtered if DEMO_MODE else None,
    )

    # ── Step 1 ───────────────────────────────────────────────────────────
    print("\n[Step 1] DataProvider.get_info() 호출")
    t0 = time.time()
    trading_info = dp1.get_info()
    elapsed = time.time() - t0
    print(f"  반환 캔들   : {len(trading_info)}개  ({elapsed:.2f}초)")
    if trading_info:
        top3 = [(c["market"], f"{c['rise_ratio']*100:.4f}%")
                for c in trading_info[:3]]
        print(f"  상위 3종목  : {top3}")

    # ── Step 2 ───────────────────────────────────────────────────────────
    print("\n[Step 2] Strategy.update_trading_info(info) 호출")
    strategy1.update_trading_info(trading_info)
    print(f"  갱신된 랭킹 : {strategy1.rankings[:5]}")
    print(f"  종가 캐시   : {len(strategy1._last_prices)}종목")

    # ── Step 3 ───────────────────────────────────────────────────────────
    print("\n[Step 3] Strategy.get_request() 호출")
    requests1 = strategy1.get_request()
    if requests1:
        print(f"  생성된 요청 : {len(requests1)}개")
        for req in requests1:
            print(
                f"    [{req['type'].upper()}] market={req['market']}  "
                f"price={req['price']:,.2f}  amount={req['amount']:.6f}"
            )
    else:
        print("  요청 없음 (랭킹 부족 또는 조건 미충족)")

    # ── Step 4 + 5 ───────────────────────────────────────────────────────
    print("\n[Step 4] Trader.send_request() + [Step 5] callback → update_result()")
    if requests1:
        results1 = []

        def _cb1(result):
            strategy1.update_result(result)
            results1.append(result)
            status = "성공" if result.get("msg") == "success" else result.get("msg")
            print(
                f"  callback → [{result['type'].upper()}] {result.get('market','')}  "
                f"price={result['price']:,.2f}  amount={result['amount']:.6f}  {status}"
            )

        trader1.send_request(requests1, _cb1)

        print(f"\n[최종 상태]")
        print(f"  잔고      : {strategy1.balance:,} KRW")
        print(f"  보유 종목 : {list(strategy1.holdings.keys())}")
    else:
        print("  요청이 없어 Step 4/5 생략")

    print("\n1턴 완료")

except Exception as e:
    nb_log.flush_on_error("1턴 실행 실패")
    raise


In [ ]:
# ── 다수 턴 시뮬레이션 ─────────────────────────────────────────────────────
print(f"{N_TURNS}턴 시뮬레이션 시작")
print(f"(DEMO_MODE={DEMO_MODE}, 실시간 모드이면 각 턴 사이 5초 대기)")
print("=" * 70)

nb_log.clear()

strategy_sim = StrategyKRWRising()
strategy_sim.initialize(INITIAL_BUDGET, min_price=1000)
trader_sim   = MockMultiTrader(initial_budget=INITIAL_BUDGET)
dp_sim       = KRWRisingDataProvider(
    min_price=MIN_PRICE, max_price=MAX_PRICE,
    candle_count=CANDLE_COUNT, top_n=TOP_N,
    demo_mode=DEMO_MODE,
    demo_base_prices=price_filtered if DEMO_MODE else None,
)

turn_records = []   # 턴별 스냅샷 [{turn, balance, holdings_val, total_asset, ...}]

try:
    for turn in range(1, N_TURNS + 1):
        ts = datetime.now().strftime("%H:%M:%S")
        print(f"\n--- Turn {turn}/{N_TURNS}  ({ts}) ---")

        info, reqs, results = _execute_trading_sync(
            dp_sim, strategy_sim, trader_sim
        )

        # 보유 자산 평가액
        holdings_val = sum(
            h["current_price"] * h["amount"]
            for h in strategy_sim.holdings.values()
        )
        total_asset = strategy_sim.balance + holdings_val

        turn_records.append({
            "turn":         turn,
            "time":         ts,
            "balance":      strategy_sim.balance,
            "holdings_val": round(holdings_val),
            "total_asset":  round(total_asset),
            "n_holdings":   len(strategy_sim.holdings),
            "n_rising":     len(info),
            "n_requests":   len(reqs) if reqs else 0,
            "n_results":    len(results),
        })

        # 체결 요약
        executed = [
            f"[{r['type'].upper()}]{r.get('market','')}@{r['price']:,.0f}"
            for r in results if r.get("msg") == "success"
        ]

        print(
            f"  상승 종목: {len(info):2d}개  |  "
            f"요청: {len(reqs) if reqs else 0}개  |  "
            f"체결: {len(executed)}건  |  "
            f"보유: {len(strategy_sim.holdings)}종목  |  "
            f"잔고: {strategy_sim.balance:,}  |  "
            f"총자산: {round(total_asset):,} KRW"
        )
        if executed:
            print(f"  체결 내역: {' | '.join(executed)}")

        # 실시간 모드: 다음 1분봉을 기다리는 대신 짧게 대기
        if not DEMO_MODE and turn < N_TURNS:
            print("  5초 후 다음 턴...")
            time.sleep(5)

except Exception as e:
    nb_log.flush_on_error("다수 턴 시뮬레이션 실패")
    raise

print(f"\n시뮬레이션 완료 ({N_TURNS}턴)")


## 섹션 7 — 결과 분석 및 시각화


In [ ]:
# ── 체결 내역 ────────────────────────────────────────────────────────────────
if trader_sim.trade_log:
    df_trades = pd.DataFrame(trader_sim.trade_log)
    df_trades["price"]  = df_trades["price"].apply(lambda x: f"{x:,.2f}")
    df_trades["total"]  = df_trades["total"].apply(lambda x: f"{x:,}")
    df_trades["fee"]    = df_trades["fee"].apply(lambda x: f"{x:,}")
    print("체결 내역:")
    display(
        df_trades[["time","type","market","price","amount","total","fee"]]
        .to_string(index=False)
    )
else:
    print("체결 내역 없음")
    print("  원인: 랭킹 종목의 지속 상승 조건이 연속 턴에서 충족되지 않았거나")
    print("        잔고 부족, DEMO_MODE 의 랜덤 데이터 특성 때문일 수 있습니다.")


In [ ]:
# ── 최종 수익률 요약 ─────────────────────────────────────────────────────────
holdings_final = sum(
    h["current_price"] * h["amount"]
    for h in strategy_sim.holdings.values()
)
total_final  = strategy_sim.balance + holdings_final
profit       = total_final - INITIAL_BUDGET
return_pct   = profit / INITIAL_BUDGET * 100

print("=" * 60)
print(" 최종 결과 요약")
print("=" * 60)
print(f"  초기 예산       : {INITIAL_BUDGET:>12,} KRW")
print(f"  최종 현금 잔고  : {strategy_sim.balance:>12,} KRW")
print(f"  보유 자산 평가  : {holdings_final:>12,.0f} KRW")
print(f"  총 자산 평가    : {total_final:>12,.0f} KRW")
print(f"  손익            : {profit:>+12,.0f} KRW")
print(f"  수익률          : {return_pct:>+12.4f} %")
print(f"  체결 건수       : {len(trader_sim.trade_log)}건")

if strategy_sim.holdings:
    print(f"\n  현재 보유 종목 ({len(strategy_sim.holdings)}개):")
    for sym, h in strategy_sim.holdings.items():
        pnl_pct = (h["current_price"] - h["avg_price"]) / h["avg_price"] * 100
        print(
            f"    {sym:<6s}  {h['amount']:.6f}개  "
            f"평균가={h['avg_price']:,.2f}  현재가={h['current_price']:,.2f}  "
            f"수익률={pnl_pct:+.2f}%"
        )
else:
    print("\n  현재 보유 종목 없음")


In [ ]:
if turn_records:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(
        f"StrategyKRWRising 시뮬레이션 결과  ({N_TURNS}턴)",
        fontsize=13, fontweight="bold"
    )

    turns = [r["turn"] for r in turn_records]

    # 1. 총 자산 변화
    ax1 = axes[0, 0]
    ax1.plot(turns, [r["total_asset"] for r in turn_records],
             "b-o", linewidth=2, markersize=5, label="총 자산")
    ax1.axhline(INITIAL_BUDGET, color="gray", linestyle="--",
                linewidth=1, label="초기 예산")
    ax1.set_title("총 자산 변화 (KRW)")
    ax1.set_xlabel("턴")
    ax1.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{x:,.0f}")
    )
    ax1.grid(alpha=0.3)
    ax1.legend(fontsize=9)

    # 2. 현금 잔고 vs 보유 자산 (누적 영역)
    ax2 = axes[0, 1]
    ax2.stackplot(
        turns,
        [r["balance"] for r in turn_records],
        [r["holdings_val"] for r in turn_records],
        labels=["현금 잔고", "보유 자산 평가"],
        colors=["#4CAF50", "#2196F3"],
        alpha=0.75,
    )
    ax2.set_title("현금 vs 보유 자산 구성")
    ax2.set_xlabel("턴")
    ax2.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{x:,.0f}")
    )
    ax2.legend(loc="upper left", fontsize=9)
    ax2.grid(alpha=0.3)

    # 3. 탐지된 지속 상승 종목 수
    ax3 = axes[1, 0]
    ax3.bar(turns, [r["n_rising"] for r in turn_records],
            color="#FF9800", alpha=0.8, edgecolor="white")
    ax3.set_title("턴별 탐지된 지속 상승 종목 수")
    ax3.set_xlabel("턴")
    ax3.set_ylabel("종목 수")
    ax3.grid(axis="y", alpha=0.3)

    # 4. 보유 종목 수 + 요청 건수
    ax4 = axes[1, 1]
    ax4.plot(turns, [r["n_holdings"] for r in turn_records],
             "r-s", linewidth=2, markersize=6, label="보유 종목 수")
    ax4.bar(turns, [r["n_requests"] for r in turn_records],
            alpha=0.35, color="purple", label="요청 건수")
    ax4.set_title("보유 종목 수 / 요청 건수")
    ax4.set_xlabel("턴")
    ax4.set_yticks(range(0, strategy_sim.MAX_BUY_COUNT + 3))
    ax4.legend(fontsize=9)
    ax4.grid(alpha=0.3)

    plt.tight_layout()
    os.makedirs("output", exist_ok=True)
    plt.savefig("output/s7_simulation_result.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("차트 저장: output/s7_simulation_result.png")
else:
    print("시각화할 데이터 없음 (시뮬레이션이 실행되지 않았습니다)")
